In [3]:
from dotenv import load_dotenv
import os
import kagglehub
import pandas
import kagglehub
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.metrics import roc_auc_score, precision_recall_curve, auc
import xgboost as xgb
import lightgbm as lgb
import shap
import matplotlib.pyplot as plt


Setup (excluding the millions of imports above, bc its python)

In [4]:
#This is a token check, you will need your own kaggle api token, refer to the README (if I remember to make one lol!)
load_dotenv('kaggletoken.env')
token = os.getenv('KAGGLE_API_TOKEN')

if not token:
    raise RuntimeError("KAGGLE_API_TOKEN not found — check kaggletoken.env")
os.environ['KAGGLE_API_TOKEN'] = token

Dataset prep

In [5]:
path = kagglehub.competition_download('home-credit-default-risk')
csv_path = path + r"/application_train.csv"

df = pd.read_csv(csv_path)

You dont have to do this, but I do this js to kinda see what im working with

In [6]:
print(df.head())
print(df.shape)
print(df.dtypes.value_counts())
pd.set_option('display.max_columns', None)
print(df.columns.tolist())

   SK_ID_CURR  TARGET NAME_CONTRACT_TYPE CODE_GENDER FLAG_OWN_CAR  \
0      100002       1         Cash loans           M            N   
1      100003       0         Cash loans           F            N   
2      100004       0    Revolving loans           M            Y   
3      100006       0         Cash loans           F            N   
4      100007       0         Cash loans           M            N   

  FLAG_OWN_REALTY  CNT_CHILDREN  AMT_INCOME_TOTAL  AMT_CREDIT  AMT_ANNUITY  \
0               Y             0          202500.0    406597.5      24700.5   
1               N             0          270000.0   1293502.5      35698.5   
2               Y             0           67500.0    135000.0       6750.0   
3               Y             0          135000.0    312682.5      29686.5   
4               Y             0          121500.0    513000.0      21865.5   

   ...  FLAG_DOCUMENT_18 FLAG_DOCUMENT_19 FLAG_DOCUMENT_20 FLAG_DOCUMENT_21  \
0  ...                 0             

In [7]:
categorical_cols = df.select_dtypes(include='str').columns.tolist()
numerical_cols = df.select_dtypes(include='number').columns.tolist()

This one is an actual real world dataset, so its gonna be a little messy

In [8]:
missing = df.isnull().sum().sort_values(ascending=False)
missing_pct = (df.isnull().sum() / len(df) * 100).sort_values(ascending=False)
print(missing_pct[missing_pct > 0])

COMMONAREA_AVG              69.872297
COMMONAREA_MODE             69.872297
COMMONAREA_MEDI             69.872297
NONLIVINGAPARTMENTS_MEDI    69.432963
NONLIVINGAPARTMENTS_MODE    69.432963
                              ...    
EXT_SOURCE_2                 0.214626
AMT_GOODS_PRICE              0.090403
AMT_ANNUITY                  0.003902
CNT_FAM_MEMBERS              0.000650
DAYS_LAST_PHONE_CHANGE       0.000325
Length: 67, dtype: float64


Turns out some of our data is missing! We will drop some of these (because they dont matter as much anyways) but we will need the exit sources

In [9]:
# Step 2: drop high-missing columns (keep EXT_SOURCE_1)
threshold = 40
cols_to_drop = missing_pct[missing_pct > threshold].index.tolist()
if 'EXT_SOURCE_1' in cols_to_drop:
    cols_to_drop.remove('EXT_SOURCE_1')
df = df.drop(columns=cols_to_drop)
print(f"Dropped {len(cols_to_drop)} columns")

Dropped 48 columns


Ok so this compeition is old, there's a huge anomaly here in the DAYS_EMPLOYED columns (I looked it up on Google btw lol)

In [10]:
df['DAYS_EMPLOYED_ANOM'] = df['DAYS_EMPLOYED'] == 365243
df['DAYS_EMPLOYED'] = df['DAYS_EMPLOYED'].replace(365243, np.nan)

Finally our X and Y loll

In [11]:
y = df['TARGET']
X = df.drop(columns=['TARGET', 'SK_ID_CURR'])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [12]:
categorical_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

numeric_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='median'))
])

preprocessor = ColumnTransformer([
    ('cat', categorical_pipe, categorical_cols),
    ('num', numeric_pipe, numerical_cols)
])

In [14]:
X_train_preprocessed = preprocessor.fit_transform(X_train)
X_test_preprocessed = preprocessor.transform(X_test)

ValueError: A given column is not a column of the dataframe